In [1]:
import torch
from transformers import AutoModel, AutoTokenizer,AutoModelForMaskedLM, AutoModelForCausalLM,AutoModelForSeq2SeqLM
import pubchempy as pcp
from scipy.io import loadmat
from utils.helpers import *
from rdkit import Chem


/Users/farzaneh/opt/anaconda3/envs/MoLFormer_fMRI/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
base_dir = '../../../../T5 EVO'
mid_dir = 'data'

In [3]:
set_seeds(seed=2024)

In [4]:
def extract_representations(tokenizer, model,model_name,input_type='smiles',token=0):
    model.eval()  
    for subject_id in range(1, 4):
        input_molecules = pd.read_csv(f'{base_dir}/fmri/embeddings/CIDs_smiles_selfies_{subject_id}.csv')[input_type].values.tolist()
        inputs = tokenizer(input_molecules, padding=True, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs,output_hidden_states=True)
            for i,output in enumerate(outputs.hidden_states):
                np.save(f'{base_dir}/fmri/embeddings/embeddings_{model_name}_{subject_id}_{i}.npy', output[:,token,:].cpu().numpy())

In [5]:
for subject_id in range(1, 4):
    CIDs, smiles = read_CIDs(base_dir,subject_id)
    selfies = smiles_to_selfies(smiles) 
    pd.DataFrame({'CIDs':CIDs, 'smiles':smiles, 'selfies':selfies}).to_csv(f'{base_dir}/fmri/embeddings/CIDs_smiles_selfies_{subject_id}.csv')

In [ ]:
for subject_id in range(1, 4):
    mat1 = loadmat(f'{base_dir}/fmri/{data}/behavior/behav_ratings_NEMO0{subject_id}.mat')
    ratings = mat1['behav'][0][0]['ratings']
    print(ratings.shape)
    #save the ratings
    np.save(f'{base_dir}/fmri/embeddings/embeddings_behavior_{subject_id}_1.npy', ratings)

(160, 18)
(160, 18)
(160, 18)


# OpenPOM

In [7]:
# model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
# tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)
all_outputs = []
layers = []
all_subjects = []
for subject_id in range(1, 4):
    CIDs=pd.read_csv(f'{base_dir}/fmri/embeddings/CIDs_smiles_selfies_{subject_id}.csv')['CIDs'].values
    # smiles_subject = pd.read_csv(f'{base_dir}/fmri/embeddings/CIDs_smiles_{subject_id}.csv')['smiles'].values
    with torch.no_grad():
        outputs = read_pom(base_dir, CIDs)
        print(outputs.shape)
        np.save(f'{base_dir}/fmri/embeddings/embeddings_openpom_{subject_id}_1.npy', outputs)
        # save the output and layers


(160, 256)
(160, 256)
(160, 256)


# Encoder-Only

## MoLFormer-XL-both-10pct

In [8]:
model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)
extract_representations(tokenizer, model,'MoLFormer-XL-both-10pct')

# all_outputs = []
# layers = []
# all_subjects = []
# for subject_id in range(1, 4):
#     CIDs, smiles_subject = read_CIDs(subject_id)
#     inputs = tokenizer(smiles_subject, padding=True, return_tensors="pt")
#     with torch.no_grad():
#         outputs = model(**inputs,output_hidden_states=True)
#         for i,output in enumerate(outputs.hidden_states):
#             #save to npy
#             np.save(f'results/embeddings_MoLFormer-XL-both-10pct_{subject_id}_{i}.npy', output[:,0,:].cpu().numpy())
        # save the output and layers


## ChemBERTa-zinc-base-v1

In [9]:
# Load model directly
tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
model = AutoModelForMaskedLM.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
extract_representations(tokenizer, model,'ChemBERTa-zinc-base-v1')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Some weights of the model checkpoint at seyonec/ChemBERTa-zinc-base-v1 were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a B

## SELFormer

In [10]:
tokenizer = AutoTokenizer.from_pretrained("HUBioDataLab/SELFormer")
model = AutoModelForMaskedLM.from_pretrained("HUBioDataLab/SELFormer")
extract_representations(tokenizer, model,'SELFormer',input_type='selfies')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## ChemBERT_ChEMBL_pretrained

In [11]:
tokenizer = AutoTokenizer.from_pretrained("jonghyunlee/ChemBERT_ChEMBL_pretrained")
model = AutoModel.from_pretrained("jonghyunlee/ChemBERT_ChEMBL_pretrained")
extract_representations(tokenizer, model,'ChemBERT_ChEMBL_pretrained')


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Some weights of BertModel were not initialized from the model checkpoint at jonghyunlee/ChemBERT_ChEMBL_pretrained and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Decoder-Only

## BARTSmiles

In [12]:
model_path = "gayane/"
model_name = "BARTSmiles"  # Replace with actual model name if different
model = AutoModel.from_pretrained(model_path+model_name)
tokenizer = AutoTokenizer.from_pretrained(model_path+model_name,add_prefix_space=True)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model.eval()
for subject_id in range(1, 4):
        
        
        input_molecules = pd.read_csv(f'{base_dir}/fmri/embeddings/CIDs_smiles_selfies_{subject_id}.csv')['smiles'].values.tolist()
        inputs = tokenizer(input_molecules, return_tensors="pt",return_token_type_ids=False, add_special_tokens=True,padding=True)
        
        with torch.no_grad():
            outputs = model(**inputs,output_hidden_states=True)
            for i,output in enumerate(outputs.decoder_hidden_states):
                np.save(f'{base_dir}/fmri/embeddings/embeddings_decoder_{model_name}_{subject_id}_{i}.npy', output[:,-1,:].cpu().numpy())
                print(i,output.shape)
                # np.save(f'{base_dir}/fmri/results/decoder_{model_name}_{subject_id}_{i}_avg.npy', output[:,:].cpu().numpy())


            for i,output in enumerate(outputs.encoder_hidden_states):
                np.save(f'{base_dir}/fmri/embeddings/embeddings_encoder_{model_name}_{subject_id}_{i}.npy', output[:,0,:].cpu().numpy())
                print(i,output.shape)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


0 torch.Size([160, 38, 1024])
1 torch.Size([160, 38, 1024])
2 torch.Size([160, 38, 1024])
3 torch.Size([160, 38, 1024])
4 torch.Size([160, 38, 1024])
5 torch.Size([160, 38, 1024])
6 torch.Size([160, 38, 1024])
7 torch.Size([160, 38, 1024])
8 torch.Size([160, 38, 1024])
9 torch.Size([160, 38, 1024])
10 torch.Size([160, 38, 1024])
11 torch.Size([160, 38, 1024])
12 torch.Size([160, 38, 1024])
0 torch.Size([160, 38, 1024])
1 torch.Size([160, 38, 1024])
2 torch.Size([160, 38, 1024])
3 torch.Size([160, 38, 1024])
4 torch.Size([160, 38, 1024])
5 torch.Size([160, 38, 1024])
6 torch.Size([160, 38, 1024])
7 torch.Size([160, 38, 1024])
8 torch.Size([160, 38, 1024])
9 torch.Size([160, 38, 1024])
10 torch.Size([160, 38, 1024])
11 torch.Size([160, 38, 1024])
12 torch.Size([160, 38, 1024])
0 torch.Size([160, 30, 1024])
1 torch.Size([160, 30, 1024])
2 torch.Size([160, 30, 1024])
3 torch.Size([160, 30, 1024])
4 torch.Size([160, 30, 1024])
5 torch.Size([160, 30, 1024])
6 torch.Size([160, 30, 1024])
7 to

## SMILES-GPT

In [13]:
from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast
model_name= 'smiles-gpt'
model_dir = f'{base_dir}/fmri/models/smiles-gpt/'
checkpoint = "checkpoints/benchmark-5m"

config = GPT2Config.from_pretrained(model_dir+checkpoint, output_hidden_states=True)
model = GPT2LMHeadModel.from_pretrained(model_dir+checkpoint, config=config)
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_dir+checkpoint,add_prefix_space=True)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model.resize_token_embeddings(len(tokenizer))
model.eval()
for subject_id in range(1, 4):
        input_molecules = pd.read_csv(f'{base_dir}/fmri/embeddings/CIDs_smiles_selfies_{subject_id}.csv')['smiles'].values.tolist()
        inputs = tokenizer(input_molecules, return_tensors="pt", add_special_tokens=True,return_token_type_ids=False,padding=True)

        with torch.no_grad():
            outputs = model(**inputs,return_dict=True)
            for i,output in enumerate(outputs.hidden_states):
                np.save(f'{base_dir}/fmri/embeddings/embeddings_{model_name}_{subject_id}_{i}.npy', output[:,-1,:].cpu().numpy())
                

/Users/farzaneh/opt/anaconda3/envs/MoLFormer_fMRI/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:818: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


# MoLGen

In [14]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# model_path = "zjunlp/"
# model_name = "MolGen-large"
# tokenizer = AutoTokenizer.from_pretrained(model_path+model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_path+model_name)
# model.eval()
# for subject_id in range(1, 4):
#         CIDs, smiles_subject = read_CIDs(base_dir,subject_id)
#         # smiles_subject2=['CC(C)CC1=CC=C(C=C1)C(C)C(=O)O','CC(C)CC1=CC=C(C=C1)C(C)C(=O']
#         inputs = tokenizer(smiles_subject, return_tensors="pt",return_token_type_ids=False, add_special_tokens=True,padding=True)
#         # inputs.pop("token_type_ids", None)
#         # print(type(smiles_subject), smiles_subject)
#         print(tokenizer.vocab_size)
#         print(tokenizer.tokenize(smiles_subject[0]))


#         with torch.no_grad():
#             outputs = model(**inputs,output_hidden_states=True)
#             for i,output in enumerate(outputs.decoder_hidden_states):
#                 print(i,output.shape)
#                 np.save(f'results/embeddings_decoder_{model_name}_{subject_id}_{i}.npy', output[:,-1,:].cpu().numpy())
#                 output = torch.mean(output, dim=1)
#                 np.save(f'results/embeddings_decoder_{model_name}_{subject_id}_{i}_avg.npy', output[:,:].cpu().numpy())


#             for i,output in enumerate(outputs.encoder_hidden_states):
#                 print(i,output.shape)
#                 np.save(f'results/embeddings_encoder_{model_name}_{subject_id}_{i}.npy', output[:,-1,:].cpu().numpy())
#                 output = torch.mean(output, dim=1)
#                 np.save(f'results/embeddings_encoder_{model_name}_{subject_id}_{i}_avg.npy', output[:,:].cpu().numpy())


## ChemGPT

In [15]:
tokenizer = AutoTokenizer.from_pretrained("ncfrey/ChemGPT-4.7M")
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model = AutoModelForCausalLM.from_pretrained("ncfrey/ChemGPT-4.7M")
extract_representations(tokenizer, model,'ChemGPT-4.7M',token=-1)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
tokenizer = AutoTokenizer.from_pretrained("ncfrey/ChemGPT-19M")
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model = AutoModelForCausalLM.from_pretrained("ncfrey/ChemGPT-19M")
extract_representations(tokenizer, model,'ChemGPT-19M',token=-1)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:

tokenizer = AutoTokenizer.from_pretrained("ncfrey/ChemGPT-1.2B")
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model = AutoModelForCausalLM.from_pretrained("ncfrey/ChemGPT-1.2B")
extract_representations(tokenizer, model,'ChemGPT-1.2B',token=-1)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# MoLGPT

In [18]:
tokenizer = AutoTokenizer.from_pretrained("msb-roshan/molgpt")
model = AutoModelForCausalLM.from_pretrained("msb-roshan/molgpt")
extract_representations(tokenizer, model,'molgpt',token=-1)

# IBM SmallMoleculeMultiViewModel

# GTMGC

# Molecular Representations

In [19]:
descriptors =pd.read_csv(f'{base_dir}/fmri/molecular_descriptors_data.txt', sep='\t')
descriptors.set_index('CID', inplace=True)
descriptors.sort_values(by='CID',inplace=True)
descriptors.fillna(value=0,inplace=True)
for subject_id in range(1, 4):
    CIDs - pd.read_csv(f'{base_dir}/fmri/embeddings/CIDs_smiles_selfies_{subject_id}.csv')['CIDs'].values
    descriptors_cid = descriptors.loc[CIDs]
    descriptors_numpy = descriptors_cid.to_numpy()
    np.save(f'{base_dir}/fmri/embeddings/embeddings_molecular_descriptors_{subject_id}_1.npy', descriptors_numpy)

    #

In [20]:

#convert dataframe to numpy array


In [21]:
descriptors_cid

,complexity from pubmed,MW,AMW,Sv,Se,Sp,Si,Mv,Me,Mp,...,Psychotic-80,Psychotic-50,Hypertens-80,Hypertens-50,Hypnotic-80,Hypnotic-50,Neoplastic-80,Neoplastic-50,Infective-80,Infective-50
CID,,,,,,,,,,,,,,,,,,,,,
240,72.5,106.13,7.581,9.295,13.978,9.739,15.455,0.664,0.998,0.696,...,0,0,0,0,0,0,0,0,0,0
261,24.8,72.12,5.548,6.822,12.862,7.500,14.870,0.525,0.989,0.577,...,0,0,0,0,0,0,0,0,0,0
263,13.1,74.14,4.943,7.349,14.745,8.262,17.285,0.490,0.983,0.551,...,0,0,0,0,0,0,0,0,0,0
325,101.0,150.24,6.010,14.402,24.513,15.784,28.116,0.576,0.981,0.631,...,0,0,0,0,0,0,0,0,1,0
326,121.0,148.22,6.444,13.876,22.629,15.023,25.701,0.603,0.984,0.653,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
643820,150.0,154.28,5.320,15.456,28.280,17.307,32.946,0.533,0.975,0.597,...,0,0,0,0,0,0,0,0,1,0
3578033,134.0,152.21,6.618,13.590,22.956,14.477,25.910,0.591,0.998,0.629,...,0,0,0,0,0,0,0,0,0,0
5365027,76.8,142.27,5.081,14.456,27.280,16.307,31.946,0.516,0.974,0.582,...,0,0,0,0,0,0,0,0,0,0


In [22]:
descriptors_numpy

array([[ 72.5  , 106.13 ,   7.581, ...,   0.   ,   0.   ,   0.   ],
       [ 24.8  ,  72.12 ,   5.548, ...,   0.   ,   0.   ,   0.   ],
       [ 13.1  ,  74.14 ,   4.943, ...,   0.   ,   0.   ,   0.   ],
       ...,
       [ 76.8  , 142.27 ,   5.081, ...,   0.   ,   0.   ,   0.   ],
       [255.   , 296.6  ,   4.862, ...,   0.   ,   0.   ,   0.   ],
       [101.   , 146.21 ,   6.092, ...,   0.   ,   0.   ,   0.   ]])